In [ ]:
# ── Cell 1: GPU check ────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), 'No GPU! Enable T4 in Kaggle Settings → Accelerator'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('PyTorch:', torch.__version__)

In [ ]:
# ── Cell 2: paths ────────────────────────────────────────────────────────────
import os, sys

CHECKPOINT_DIR = '/kaggle/working/checkpoints'
DATA_DIR       = '/kaggle/working/data'
CODE_DIR       = '/kaggle/working/code'

# Define ALL derived paths here so they're available after any kernel restart
CUBICASA_RAW       = f'{DATA_DIR}/raw/cubicasa5k'
CUBICASA_REPO      = '/kaggle/working/cubicasa_repo'
CUBICASA_PROCESSED = f'{DATA_DIR}/processed/cubicasa'
PSEUDO_PROCESSED   = f'{DATA_DIR}/processed/pseudo12k'
CVCFP_PROCESSED    = f'{DATA_DIR}/processed/cvcfp'
RESPLAN_PROCESSED  = f'{DATA_DIR}/processed/resplan'
COMBINED_DIR       = f'{DATA_DIR}/processed/combined/splits'

for d in [CHECKPOINT_DIR, DATA_DIR]:
    os.makedirs(d, exist_ok=True)
print('Paths ready')

In [ ]:
# ── Cell 3: install packages ──────────────────────────────────────────────────
# FIX: pin smp version — the MiT-B2 encoder transplant is verified against 0.3.3
# Newer versions change internal encoder APIs and will break the transplant.
!pip install -q 'segmentation-models-pytorch==0.3.3' albumentations huggingface_hub datasets transformers lxml svgpathtools
# Verify smp installed correctly
import segmentation_models_pytorch as smp
print('smp version:', smp.__version__)

In [ ]:
# ── Cell 4: clone / pull code repo ───────────────────────────────────────────
import os, sys

# FIX: check for .git directory specifically to detect a proper clone
if not os.path.isdir(f'{CODE_DIR}/.git'):
    ret = os.system(f'git clone --branch v3-playground https://github.com/Shital-P276/6-MP.git {CODE_DIR}')
    assert ret == 0, 'git clone failed — check repo URL and branch name'
else:
    os.system(f'git -C {CODE_DIR} pull origin v3-playground')

# Add src and tools to path
for p in [f'{CODE_DIR}/ml/src', f'{CODE_DIR}/ml/tools']:
    if p not in sys.path:
        sys.path.insert(0, p)

# Verify key files are present
required = ['model.py', 'loss.py', 'dataset.py', 'train.py', 'evaluate.py']
missing  = [f for f in required if not os.path.exists(f'{CODE_DIR}/ml/src/{f}')]
if missing:
    raise FileNotFoundError(f'Missing source files: {missing} — did the git push succeed?')
print('Source files verified:', required)

In [ ]:
# ── Cell 5: mock TensorBoard BEFORE any imports (avoids TF conflict on Kaggle)
import sys, unittest.mock as mock
sys.modules['torch.utils.tensorboard'] = mock.MagicMock()
sys.modules['tensorboard'] = mock.MagicMock()
print('TensorBoard mocked')

In [ ]:
# ── Cell 6: verify model builds correctly ────────────────────────────────────
# Run this BEFORE spending time downloading data — catches version issues immediately
from model import build_mitunet, get_param_groups
import torch

print('Building MitUNet (pretrained=False for verification)...')
m = build_mitunet(num_classes=4, pretrained=False)
x = torch.randn(1, 3, 512, 512)
with torch.no_grad():
    out = m(x)
assert out.shape == (1, 4, 512, 512), f'Wrong output shape: {out.shape}'
total = sum(p.numel() for p in m.parameters()) / 1e6
enc_p, dec_p = get_param_groups(m)
print(f'Model OK ✓  |  {total:.1f}M params  |  encoder={len(enc_p)} param tensors, decoder={len(dec_p)}')
del m, x, out

In [ ]:
# ── Cell 7: HuggingFace token ────────────────────────────────────────────────
# Must be toggled ON in Kaggle Add-ons → Secrets before running
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    HF_REPO  = 'Shital-P276/floorviz-mitunet-b2'
    print(f'HF token loaded ✓  Repo: {HF_REPO}')
except Exception as e:
    print(f'HF_TOKEN not available ({e})')
    print('Checkpoints will be saved locally only — upload manually after training')
    HF_TOKEN = None
    HF_REPO  = None

In [ ]:
# ── Cell 8: load CubiCasa5k ──────────────────────────────────────────────────
import os

if not os.path.exists(CUBICASA_RAW):
    os.makedirs(f'{DATA_DIR}/raw', exist_ok=True)

    # Check for Kaggle-attached dataset first — avoids re-downloading every run
    # Path matches the double-nested structure: /kaggle/input/cubicasa5k/cubicasa5k/
    KAGGLE_CUBI = '/kaggle/input/cubicasa5k/cubicasa5k'
    if os.path.exists(KAGGLE_CUBI):
        print('CubiCasa5k found as attached Kaggle dataset — symlinking...')
        os.symlink(KAGGLE_CUBI, CUBICASA_RAW)
        print('Symlink created. No download needed.')
    else:
        print('Downloading CubiCasa5k (~5GB, ~15 mins)...')
        ret = os.system(f'wget -q --show-progress "https://zenodo.org/record/2613548/files/cubicasa5k.zip" -O /kaggle/working/cubicasa5k.zip')
        assert ret == 0, 'wget failed — check Zenodo URL is still live'
        os.system(f'unzip -q /kaggle/working/cubicasa5k.zip -d {DATA_DIR}/raw/')
        # BUG 10 fix: handle double-nested unzip (cubicasa5k/cubicasa5k/)
        nested = f'{DATA_DIR}/raw/cubicasa5k/cubicasa5k'
        if os.path.exists(nested):
            print('Flattening double-nested unzip...')
            os.system(f'mv {nested}/* {DATA_DIR}/raw/cubicasa5k/')
            os.system(f'rmdir {nested}')
        os.system('rm /kaggle/working/cubicasa5k.zip')
        print('Download complete.')
else:
    print('CubiCasa5k already present.')

if not os.path.exists(CUBICASA_REPO):
    print('Cloning CubiCasa5k loader repo...')
    os.system(f'git clone --depth 1 https://github.com/CubiCasa/CubiCasa5k.git {CUBICASA_REPO}')
else:
    print('CubiCasa repo already present.')

# Quick check
subdirs = [d for d in os.listdir(CUBICASA_RAW) if os.path.isdir(f'{CUBICASA_RAW}/{d}')]
print(f'CubiCasa5k subfolders: {len(subdirs)} (expected 3)  e.g. {subdirs[:3]}')

In [ ]:
# ── Cell 9: prepare CubiCasa5k masks ─────────────────────────────────────────
import os, json

if not os.path.exists(f'{CUBICASA_PROCESSED}/splits/train.json'):
    print('Generating CubiCasa5k masks (this takes ~20-40 mins)...')
    ret = os.system(
        f'python {CODE_DIR}/ml/tools/prepare_cubicasa.py '
        f'--cubicasa-root {CUBICASA_RAW} '
        f'--output-root {CUBICASA_PROCESSED} '
        f'--floortrans-repo {CUBICASA_REPO}'
    )
    # BUG 9 fix: soft warning instead of hard assert
    if ret != 0:
        print('WARNING: prepare_cubicasa.py exited non-zero — check output above.')
        print('Continuing — will use whatever masks were produced.')
else:
    print('CubiCasa5k already processed.')

with open(f'{CUBICASA_PROCESSED}/splits/train.json') as f:
    cubi_train = json.load(f)
with open(f'{CUBICASA_PROCESSED}/splits/val.json') as f:
    cubi_val = json.load(f)
print(f'CubiCasa5k — train: {len(cubi_train)}  val: {len(cubi_val)}')

# BUG 9 fix: soft warning, not hard assert
if len(cubi_train) < 500:
    print(f'WARNING: Only {len(cubi_train)} CubiCasa train samples.')
    print('Stage 1 will fall back to combined dataset as per STAGE1_CONFIG fallback logic.')
elif len(cubi_train) < 3000:
    print(f'WARNING: Only {len(cubi_train)} train samples — lower than expected but continuing.')
else:
    print(f'CubiCasa5k looks good ✓')

In [ ]:
# ── Cell 10: prepare pseudo-floor-plan-12k ────────────────────────────────────
import os, json

if not os.path.exists(f'{PSEUDO_PROCESSED}/splits/train.json'):
    print('Downloading + processing pseudo-floor-plan-12k (~30 mins)...')
    ret = os.system(
        f'python {CODE_DIR}/ml/tools/prepare_pseudo12k.py '
        f'--output-root {PSEUDO_PROCESSED}'
    )
    # BUG 9 fix: soft warning instead of hard assert
    if ret != 0:
        print('WARNING: prepare_pseudo12k.py failed — writing empty splits and continuing.')
        os.makedirs(f'{PSEUDO_PROCESSED}/splits', exist_ok=True)
        for s in ('train', 'val', 'test'):
            open(f'{PSEUDO_PROCESSED}/splits/{s}.json', 'w').write('[]')
else:
    print('pseudo-12k already processed.')

with open(f'{PSEUDO_PROCESSED}/splits/train.json') as f:
    pseudo_train = json.load(f)
print(f'pseudo-12k — train: {len(pseudo_train)}')

# BUG 9 fix: soft warning, not hard assert
if len(pseudo_train) < 8000:
    print(f'WARNING: Only {len(pseudo_train)} pseudo samples — lower than expected but continuing.')
else:
    print('pseudo-12k looks good ✓')

In [ ]:
# ── Cell: prepare CVC-FP ─────────────────────────────────────────────────────
# CVC-FP: 122 floor plans in 4 architectural styles — adds visual diversity
# Download from: https://dag.cvc.uab.es/dataset/cvc-fp-database/
# Manual step: download CVC-FP.zip, upload to Kaggle as a dataset,
# then set CVCFP_RAW to the extracted path below.
# If you don't have it, this cell skips gracefully.
import os, json

CVCFP_RAW       = '/kaggle/input/cvc-fp/CVC-FP'   # adjust if uploaded differently
CVCFP_PROCESSED = f'{DATA_DIR}/processed/cvcfp'

if not os.path.exists(CVCFP_RAW):
    print('CVC-FP not found at', CVCFP_RAW)
    print('Skipping — training will proceed without CVC-FP.')
    print('To add it: download from dag.cvc.uab.es and upload as a Kaggle dataset.')
    # Write empty splits so merge_splits.py does not crash
    os.makedirs(f'{CVCFP_PROCESSED}/splits', exist_ok=True)
    for s in ('train', 'val', 'test'):
        open(f'{CVCFP_PROCESSED}/splits/{s}.json', 'w').write('[]')
elif not os.path.exists(f'{CVCFP_PROCESSED}/splits/train.json'):
    print('Processing CVC-FP (~122 plans)...')
    ret = os.system(
        f'python {CODE_DIR}/ml/tools/prepare_cvcfp.py '
        f'--cvcfp-root {CVCFP_RAW} '
        f'--output-root {CVCFP_PROCESSED}'
    )
    assert ret == 0, 'prepare_cvcfp.py failed'
else:
    print('CVC-FP already processed.')

with open(f'{CVCFP_PROCESSED}/splits/train.json') as f:
    cvcfp_train = json.load(f)
print(f'CVC-FP train samples: {len(cvcfp_train)}')


In [ ]:
# ── Cell 11: prepare ResPlan ─────────────────────────────────────────────────
# ResPlan: 17,000 residential floor plans with walls/doors/windows/balconies
# Source: https://github.com/m-agour/ResPlan (GitHub PKL, not HuggingFace)
# Script checks for Kaggle-attached dataset first, then downloads from GitHub.
import os, json

if not os.path.exists(f'{RESPLAN_PROCESSED}/splits/train.json'):
    print('Processing ResPlan (downloads from GitHub if not attached as dataset)...')
    ret = os.system(
        f'python {CODE_DIR}/ml/tools/prepare_resplan.py '
        f'--output-root {RESPLAN_PROCESSED}'
    )
    # Non-fatal — script writes empty splits on failure
    if ret != 0:
        print('ResPlan not available — writing empty splits and continuing.')
        os.makedirs(f'{RESPLAN_PROCESSED}/splits', exist_ok=True)
        for s in ('train', 'val', 'test'):
            open(f'{RESPLAN_PROCESSED}/splits/{s}.json', 'w').write('[]')
else:
    print('ResPlan already processed.')

with open(f'{RESPLAN_PROCESSED}/splits/train.json') as f:
    resplan_train = json.load(f)
print(f'ResPlan train samples: {len(resplan_train)}')
if not resplan_train:
    print('(ResPlan not available — training will use CubiCasa5k + pseudo-12k + CVC-FP)')

In [ ]:
# ── Cell 12: merge all splits ────────────────────────────────────────────────
import os, json

if not os.path.exists(f'{COMBINED_DIR}/train.json'):
    print('Merging splits from all datasets...')
    ret = os.system(
        f'python {CODE_DIR}/ml/tools/merge_splits.py '
        f'--inputs '
        f'{DATA_DIR}/processed/cubicasa/splits '
        f'{DATA_DIR}/processed/pseudo12k/splits '
        f'{DATA_DIR}/processed/cvcfp/splits '
        f'{DATA_DIR}/processed/resplan/splits '
        f'--output {COMBINED_DIR}'
    )
    assert ret == 0, 'merge_splits.py failed'
else:
    print('Combined splits already exist.')

with open(f'{COMBINED_DIR}/train.json') as f:
    combined_train = json.load(f)
with open(f'{COMBINED_DIR}/val.json') as f:
    combined_val = json.load(f)

# Show breakdown by source
from collections import Counter
sources = Counter(s['source'] for s in combined_train)
print(f'Combined train: {len(combined_train)} total')
for src, count in sorted(sources.items()):
    print(f'  {src}: {count}')
print(f'Combined val: {len(combined_val)}')

# BUG 9 fix: soft warning
if len(combined_train) < 3000:
    print(f'WARNING: Only {len(combined_train)} combined samples — check prep steps above.')
else:
    print('Combined dataset looks good ✓')

In [ ]:
# ── Cell 12: mask sanity check ────────────────────────────────────────────────
# MUST pass before training. If any assert fails, fix the prep scripts.
import json, cv2, numpy as np, matplotlib.pyplot as plt, random

with open(f'{COMBINED_DIR}/train.json') as f:
    samples = json.load(f)

# bg=grey, wall=blue, door=red, window=yellow
CMAP = np.array([[80,80,80],[30,100,255],[255,50,50],[255,220,0]], dtype=np.uint8)

test_samples = random.sample(samples, 6)
fig, axes    = plt.subplots(2, 6, figsize=(18, 6))

errors = []
for i, s in enumerate(test_samples):
    img  = cv2.cvtColor(cv2.imread(s['image']), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(s['mask'], cv2.IMREAD_GRAYSCALE)
    assert img  is not None, f'Cannot read image: {s["image"]}'
    assert mask is not None, f'Cannot read mask:  {s["mask"]}'

    unique = np.unique(mask).tolist()
    bad_vals = set(unique) - {0, 1, 2, 3}
    if bad_vals:
        errors.append(f'INVALID mask values {bad_vals} in {s["mask"]}')
    if 1 not in unique:
        errors.append(f'NO wall pixels in {s["mask"]}')

    axes[0][i].imshow(img);          axes[0][i].axis('off'); axes[0][i].set_title(s['source'], fontsize=9)
    axes[1][i].imshow(CMAP[mask]);   axes[1][i].axis('off'); axes[1][i].set_title(str(unique), fontsize=8)

plt.suptitle('Top: image  |  Bottom: mask  (grey=bg, blue=wall, red=door, yellow=window)')
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/sanity_check.png', dpi=80, bbox_inches='tight')
plt.show()

if errors:
    for e in errors:
        print(f'ERROR: {e}')
    raise AssertionError('Mask sanity check FAILED — fix prep scripts before training')
else:
    print('Sanity check PASSED ✓ — ready to train')

In [ ]:
# ── Cell 13: Stage 1 — pre-train on CubiCasa5k (30 epochs) ──────────────────
# Self-restarting: if something crashes, just re-run this cell.
# It reads stage1_recovery.pth and continues from the last completed epoch.
# Max 5 restart attempts before giving up entirely.

import time, os
from train import train, STAGE1_CONFIG

stage1_cfg                = STAGE1_CONFIG.copy()
stage1_cfg['batch_size']  = 8
stage1_cfg['num_workers'] = 2

STAGE1_SAVE_DIR   = f'{CHECKPOINT_DIR}/stage1'
STAGE1_RECOVERY   = f'{STAGE1_SAVE_DIR}/stage1_recovery.pth'
STAGE1_BEST       = f'{STAGE1_SAVE_DIR}/stage1_best.pth'
MAX_RESTART_ATTEMPTS = 5

stage1_ckpt = None
for attempt in range(1, MAX_RESTART_ATTEMPTS + 1):
    try:
        if attempt > 1:
            print(f'\n>>> AUTO-RESTART attempt {attempt}/{MAX_RESTART_ATTEMPTS} <<<', flush=True)
            print(f'Waiting 10s before restart...', flush=True)
            time.sleep(10)
            # Clear GPU memory between restarts
            import torch, gc
            gc.collect()
            torch.cuda.empty_cache()
            print(f'GPU memory cleared. Resuming from last recovery checkpoint.', flush=True)

        stage1_ckpt = train(
            cfg        = stage1_cfg,
            train_json = f'{DATA_DIR}/processed/cubicasa/splits/train.json',
            val_json   = f'{DATA_DIR}/processed/cubicasa/splits/val.json',
            save_dir   = STAGE1_SAVE_DIR,
            hf_repo_id = HF_REPO,
            hf_token   = HF_TOKEN,
        )
        print(f'Stage 1 complete ✓  Best checkpoint: {stage1_ckpt}', flush=True)
        break   # success — exit retry loop

    except KeyboardInterrupt:
        print('\nManual stop. Emergency checkpoint already saved by train().', flush=True)
        break

    except Exception as e:
        print(f'\n[CRASH] Attempt {attempt} failed: {type(e).__name__}: {e}', flush=True)
        if attempt == MAX_RESTART_ATTEMPTS:
            print('All restart attempts exhausted.', flush=True)
            # Use whatever best checkpoint exists
            if os.path.exists(STAGE1_BEST):
                stage1_ckpt = STAGE1_BEST
                print(f'Using partial best checkpoint: {stage1_ckpt}', flush=True)
            elif os.path.exists(STAGE1_RECOVERY):
                stage1_ckpt = STAGE1_RECOVERY
                print(f'Using recovery checkpoint: {stage1_ckpt}', flush=True)
            raise
        # Otherwise loop back and restart

print(f'\nStage 1 checkpoint to use: {stage1_ckpt}')


In [ ]:
# ── Cell 14: evaluate Stage 1 ────────────────────────────────────────────────
from evaluate import evaluate_checkpoint
import os

stage1_ckpt = f'{CHECKPOINT_DIR}/stage1/stage1_best.pth'
assert os.path.exists(stage1_ckpt), f'Stage 1 checkpoint missing: {stage1_ckpt}'

metrics = evaluate_checkpoint(
    ckpt_path = stage1_ckpt,
    val_json  = f'{DATA_DIR}/processed/cubicasa/splits/val.json',
)

wall_iou = metrics['wall_iou']
print(f'\nStage 1 wall IoU: {wall_iou:.4f}')
if wall_iou < 0.55:
    print('CRITICAL: wall IoU < 0.55 — almost certainly a mask class mapping bug.')
    print('Run: import cv2, numpy as np; m = cv2.imread("some_mask.png", 0); print(np.unique(m))')
    print('Values must be [0, 1, 2, 3]. If you see [0, 255] the masks were not normalised.')
elif wall_iou < 0.76:
    print(f'Below 0.76 target. Consider running 10 more Stage 1 epochs before proceeding.')
    print('You can increase stage1_cfg["epochs"] to 40 and re-run Cell 13 — it will resume.')
else:
    print('Stage 1 target met ✓ — ready for Stage 2')

In [ ]:
# ── Cell 15: visualise Stage 1 predictions ───────────────────────────────────
import torch, cv2, numpy as np, matplotlib.pyplot as plt, random
from model   import build_mitunet
from dataset import FloorPlanDataset, get_val_transforms

device = torch.device('cuda')
state  = torch.load(f'{CHECKPOINT_DIR}/stage1/stage1_best.pth', map_location='cpu', weights_only=True)
model  = build_mitunet(num_classes=4, pretrained=False).to(device).eval()
model.load_state_dict(state['model'])

ds   = FloorPlanDataset(f'{DATA_DIR}/processed/cubicasa/splits/val.json', transforms=get_val_transforms())
CMAP = np.array([[80,80,80],[30,100,255],[255,50,50],[255,220,0]], dtype=np.uint8)

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i in range(6):
    img_t, mask_t = ds[random.randint(0, len(ds)-1)]
    with torch.no_grad():
        pred = model(img_t.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()

    # De-normalise image for display
    img_np = img_t.permute(1,2,0).numpy()
    img_np = (img_np * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])).clip(0,1)

    axes[0][i].imshow(img_np);               axes[0][i].axis('off'); axes[0][i].set_title('input')
    axes[1][i].imshow(CMAP[mask_t.numpy()]); axes[1][i].axis('off'); axes[1][i].set_title('ground truth')
    axes[2][i].imshow(CMAP[pred]);           axes[2][i].axis('off'); axes[2][i].set_title('prediction')

plt.suptitle('Stage 1: walls should be sharp blue lines, not blobs')
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/stage1_viz.png', dpi=80, bbox_inches='tight')
plt.show()
del model

In [ ]:
# ── Cell 16: Stage 2 — fine-tune on combined dataset (20 epochs) ─────────────
# Self-restarting: re-run this cell any time if it crashes.
# Continues from the last completed Stage 2 epoch automatically.

import time, os
from train import train, STAGE2_CONFIG

# Determine Stage 1 input checkpoint
STAGE1_BEST     = f'{CHECKPOINT_DIR}/stage1/stage1_best.pth'
STAGE1_RECOVERY = f'{CHECKPOINT_DIR}/stage1/stage1_recovery.pth'

if os.path.exists(STAGE1_BEST):
    s1_input = STAGE1_BEST
elif os.path.exists(STAGE1_RECOVERY):
    s1_input = STAGE1_RECOVERY
    print(f'WARNING: using Stage 1 recovery checkpoint (not best): {s1_input}', flush=True)
else:
    raise FileNotFoundError(
        f'No Stage 1 checkpoint found at {STAGE1_BEST}.\n'
        'Run Stage 1 cell first.'
    )
print(f'Stage 1 input: {s1_input}', flush=True)

stage2_cfg                = STAGE2_CONFIG.copy()
stage2_cfg['batch_size']  = 8
stage2_cfg['num_workers'] = 2

STAGE2_SAVE_DIR      = f'{CHECKPOINT_DIR}/stage2'
STAGE2_RECOVERY      = f'{STAGE2_SAVE_DIR}/stage2_recovery.pth'
STAGE2_BEST          = f'{STAGE2_SAVE_DIR}/stage2_best.pth'
MAX_RESTART_ATTEMPTS = 5

# If Stage 2 has a recovery checkpoint already, resume from it (not Stage 1)
# This handles the case where Stage 2 started, then crashed mid-run
resume_from = STAGE2_RECOVERY if os.path.exists(STAGE2_RECOVERY) else s1_input
print(f'Resuming from: {resume_from}', flush=True)

stage2_ckpt = None
for attempt in range(1, MAX_RESTART_ATTEMPTS + 1):
    try:
        if attempt > 1:
            print(f'\n>>> AUTO-RESTART attempt {attempt}/{MAX_RESTART_ATTEMPTS} <<<', flush=True)
            time.sleep(10)
            import torch, gc
            gc.collect()
            torch.cuda.empty_cache()
            # Always resume from Stage 2 recovery if it exists by now
            if os.path.exists(STAGE2_RECOVERY):
                resume_from = STAGE2_RECOVERY
            print(f'Resuming from: {resume_from}', flush=True)

        stage2_ckpt = train(
            cfg         = stage2_cfg,
            train_json  = f'{COMBINED_DIR}/train.json',
            val_json    = f'{COMBINED_DIR}/val.json',
            save_dir    = STAGE2_SAVE_DIR,
            resume_ckpt = resume_from,
            hf_repo_id  = HF_REPO,
            hf_token    = HF_TOKEN,
        )
        print(f'Stage 2 complete ✓  Best checkpoint: {stage2_ckpt}', flush=True)
        break

    except KeyboardInterrupt:
        print('\nManual stop. Emergency checkpoint already saved.', flush=True)
        break

    except Exception as e:
        print(f'\n[CRASH] Attempt {attempt} failed: {type(e).__name__}: {e}', flush=True)
        if attempt == MAX_RESTART_ATTEMPTS:
            if os.path.exists(STAGE2_BEST):
                stage2_ckpt = STAGE2_BEST
                print(f'Using partial best checkpoint: {stage2_ckpt}', flush=True)
            elif os.path.exists(STAGE2_RECOVERY):
                stage2_ckpt = STAGE2_RECOVERY
                print(f'Using recovery checkpoint: {stage2_ckpt}', flush=True)
            raise

print(f'\nStage 2 checkpoint to use: {stage2_ckpt}')


In [ ]:
# ── Cell 17: final evaluation on held-out test set ───────────────────────────
# Only run this once — DO NOT use test set for any hyperparameter decisions
from evaluate import evaluate_checkpoint

metrics = evaluate_checkpoint(
    ckpt_path = f'{CHECKPOINT_DIR}/stage2/stage2_best.pth',
    val_json  = f'{COMBINED_DIR}/test.json',
)

print('\n── Pass/Fail against targets ────────────────────')
checks = [
    ('Wall IoU',       metrics['wall_iou'],       0.75, 0.85),
    ('Wall Precision', metrics['wall_precision'],  0.88, 0.93),
    ('Wall Recall',    metrics['wall_recall'],     0.85, 0.91),
    ('Door IoU',       metrics['door_iou'],        0.55, 0.70),
    ('Window IoU',     metrics['window_iou'],      0.50, 0.65),
    ('mIoU',           metrics['miou'],            0.70, 0.82),
]
all_pass = True
for name, val, minimum, target in checks:
    if val >= target:
        status = '✓ TARGET'
    elif val >= minimum:
        status = '~ PASS'
    else:
        status = '✗ FAIL'
        all_pass = False
    print(f'  {name:<20} {val:.4f}   min={minimum}  target={target}   {status}')
print('─' * 55)
print('OVERALL:', '✓ ALL PASS' if all_pass else '✗ SOME FAILURES — see escalation guide in plan doc')

In [ ]:
# ── Cell 18: visualise Stage 2 predictions ───────────────────────────────────
import torch, cv2, numpy as np, matplotlib.pyplot as plt, random
from model   import build_mitunet
from dataset import FloorPlanDataset, get_val_transforms

device = torch.device('cuda')
state  = torch.load(f'{CHECKPOINT_DIR}/stage2/stage2_best.pth', map_location='cpu', weights_only=True)
model  = build_mitunet(num_classes=4, pretrained=False).to(device).eval()
model.load_state_dict(state['model'])

# Test on combined val (includes CAD-colored pseudo12k)
ds   = FloorPlanDataset(f'{COMBINED_DIR}/val.json', transforms=get_val_transforms())
CMAP = np.array([[80,80,80],[30,100,255],[255,50,50],[255,220,0]], dtype=np.uint8)

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i in range(6):
    img_t, mask_t = ds[random.randint(0, len(ds)-1)]
    with torch.no_grad():
        pred = model(img_t.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()
    img_np = img_t.permute(1,2,0).numpy()
    img_np = (img_np * np.array([0.229,0.224,0.225]) + np.array([0.485,0.456,0.406])).clip(0,1)
    axes[0][i].imshow(img_np);               axes[0][i].axis('off'); axes[0][i].set_title('input')
    axes[1][i].imshow(CMAP[mask_t.numpy()]); axes[1][i].axis('off'); axes[1][i].set_title('ground truth')
    axes[2][i].imshow(CMAP[pred]);           axes[2][i].axis('off'); axes[2][i].set_title('prediction')

plt.suptitle('Stage 2 final: walls=blue, doors=red, windows=yellow')
plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/stage2_viz.png', dpi=80, bbox_inches='tight')
plt.show()
del model

In [ ]:
# ── Cell 19: final HuggingFace upload ────────────────────────────────────────
# Uploads ALL training artefacts — checkpoints, logs, visualisations
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi
import os

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
api      = HfApi()
REPO_ID  = 'Shital-P276/floorviz-mitunet-b2'

api.create_repo(REPO_ID, private=True, exist_ok=True, token=HF_TOKEN)

files_to_upload = [
    (f'{CHECKPOINT_DIR}/stage1/stage1_best.pth',  'stage1_best.pth'),
    (f'{CHECKPOINT_DIR}/stage1/stage1_log.csv',   'stage1_log.csv'),
    (f'{CHECKPOINT_DIR}/stage2/stage2_best.pth',  'stage2_best.pth'),
    (f'{CHECKPOINT_DIR}/stage2/stage2_log.csv',   'stage2_log.csv'),
    (f'{CHECKPOINT_DIR}/sanity_check.png',         'sanity_check.png'),
    (f'{CHECKPOINT_DIR}/stage1_viz.png',           'stage1_viz.png'),
    (f'{CHECKPOINT_DIR}/stage2_viz.png',           'stage2_viz.png'),
]

for local_path, repo_path in files_to_upload:
    if os.path.exists(local_path):
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=repo_path,
            repo_id=REPO_ID,
            token=HF_TOKEN,
        )
        print(f'  ✓ Uploaded: {repo_path}')
    else:
        print(f'  ⚠ Not found: {local_path}')

print(f'\nAll done → https://huggingface.co/{REPO_ID}')